# Replication Starter: Hansen (2015), Punishment and Deterrence in Drunk Driving
## ECON 5200 · Midterm Replication Study · regression discontinuity

Hansen, B. (2015). "Punishment and Deterrence: Evidence from Drunk Driving." *American Economic Review* 105(4):
1581–1617.

> **AI rule for this project.** Sections 1 and 2 (positioning, direct replication) are **no AI**: not for
> the code, not for the prose. Sections 3 to 6 allow AI as a documented co-pilot under P.R.I.M.E., with the full
> transcript in `ai-appendix.md`. Every cell below is marked with the section it belongs to.

**How to use this notebook.** It is a starter, not the project. Copy it into your repository made from
[RyanPiao/econ5200-replication-template](https://github.com/RyanPiao/econ5200-replication-template), run it top to
bottom, and fill the cells marked **YOUR TURN**. Cells marked **GIVEN** run as they are; cells marked **CHECK** test
what you wrote and print a message, never the answer. A blank copy runs from top to bottom and prints no estimate:
the numbers appear once you have written Section 2.

In [ ]:
# GIVEN — setup: the two RD packages Colab does not have, then the libraries
%pip install rdrobust==2.0.0 rddensity==3.0 -q

import numpy as np
import pandas as pd
import statsmodels.api as sm
from rdrobust import rdrobust
from rddensity import rddensity

print("Setup complete.")

## The data

`hansen_dwi.dta` has one row per DUI stop in Washington State, 1999–2007: 214,558 stops. The variables the design
uses:

| variable | what it is |
|---|---|
| `recidivism` | 1 if the driver was stopped again for drunk driving within the paper's follow-up window (the outcome) |
| `bac1`, `bac2` | the two breath-test readings, in grams per decilitre (0.08 is the DUI threshold, 0.15 the aggravated one) |
| `low_score` | the lower of the two readings, **in thousandths** (80 means 0.080) |
| `male`, `white`, `aged`, `acc` | sex, race, age, and whether the stop involved an accident |
| `year`, `Date` | when the stop happened |

Two decisions are yours and belong in Section 2's discrepancy table: **which reading is the running variable**
(the paper says which one the law uses; find it), and how the paper's bandwidth, kernel and controls are set.

In [ ]:
# GIVEN — Section 2 starts here: load the data
dwi = pd.read_stata("https://raw.githubusercontent.com/scunning1975/mixtape/master/hansen_dwi.dta")
print(f"{len(dwi):,} stops, {dwi['year'].min():.0f}-{dwi['year'].max():.0f}")
dwi[["recidivism", "bac1", "bac2", "low_score", "male", "white", "aged", "acc"]].describe().round(3)

## Section 2: direct replication — YOUR TURN, no AI

Estimate the effect of crossing the 0.08 threshold on recidivism the way the paper's headline table does: a local
linear regression on each side of the cutoff within a bandwidth, with the paper's kernel and controls. Write the
regression yourself (`statsmodels`), not with `rdrobust`: the point is to match the paper's own estimator.

Set these names, because Section 3 reuses them:

- `RUNNING`: the column you use as the running variable, in the same units as `CUTOFF` (convert `low_score` if you use it)
- `H`: your bandwidth; `KERNEL`: `"rectangular"` or `"triangular"`; `CONTROLS`: a list of column names
- `rd_fit`: your fitted statsmodels result; `JUMP`: your estimate of the jump at 0.08, as a float

In [ ]:
# GIVEN
CUTOFF = 0.08

In [ ]:
# YOUR TURN — Section 2, no AI: set RUNNING, H, KERNEL, CONTROLS, fit rd_fit, and set JUMP
RUNNING = None
H = None
KERNEL = None
CONTROLS = None
rd_fit = None
JUMP = None

### Your number, with its unit

`recidivism` is 0 or 1, so its mean is a proportion and your jump is a change in a proportion. Read it in
**percentage points**: −0.02 is minus 2 percentage points, not "minus 2 percent". If you want a relative change,
name its base: the mean recidivism just below the cutoff, within your bandwidth. Fill all five names:

- `PRINTED`: the jump exactly as your code prints it
- `IN_UNITS`: the same jump in the unit you name in `UNIT`
- `UNIT`: one of `percentage points`, `percent relative to the mean below the cutoff`
- `PAPER_NUMBER`, `PAPER_WHERE`: the paper's own estimate, and its table, column and row, read from the paper

In [ ]:
# YOUR TURN — Section 2, no AI
PRINTED = None
IN_UNITS = None
UNIT = None
PAPER_NUMBER = None
PAPER_WHERE = None

In [ ]:
# CHECK — Section 2: your headline number, with its unit. Run it; you need not read it.
def check_headline(printed, in_units, unit, paper_number, paper_where, units_ok):
    """Is the unit right for this outcome, and is the in-unit number the printed one converted?"""
    if any(v is None for v in (printed, in_units, unit, paper_number, paper_where)):
        print("Not filled yet: set all five names in the cell above.")
        return False
    u = " ".join(str(unit).lower().split())
    if u not in units_ok:
        print(f"'{unit}' is not a unit this outcome is measured in. Allowed here: " + "; ".join(sorted(units_ok)) + ".")
        print("Ask what the outcome variable is: a log, a proportion (0 to 1), or a rate per 100,000.")
        return False
    wants = [f(float(printed)) for f in units_ok[u]]     # every accepted reading of this unit
    if not any(abs(float(in_units) - w) <= max(0.005, 0.02 * abs(w)) for w in wants):
        print(f"Your in-unit number does not follow from your printed estimate in '{unit}'.")
        print("Convert the printed coefficient again: what does one unit of the outcome mean here?")
        return False
    if not str(paper_where).strip() or len(str(paper_where).split()) < 2:
        print("Name where the paper's number is: table, column (or figure), and row.")
        return False
    print("Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.")
    return True

if RUNNING is None or H is None:
    BASE = None
else:
    below = dwi[(dwi[RUNNING] < CUTOFF) & (dwi[RUNNING] >= CUTOFF - H)]
    BASE = below["recidivism"].mean()                  # the mean just below the cutoff, within your bandwidth
HANSEN_UNITS = {
    "percentage points": [lambda b: 100 * b],
    "percent relative to the mean below the cutoff": [lambda b: 100 * b / BASE if BASE else float("nan")],
}
ok = check_headline(PRINTED, IN_UNITS, UNIT, PAPER_NUMBER, PAPER_WHERE, HANSEN_UNITS)

## Section 3: robustness — GIVEN code, AI allowed from here

These checks use `rdrobust` and `rddensity` (Calonico, Cattaneo and Titiunik; Cattaneo, Jansson and Ma), which
the paper predates: local linear with a data-driven bandwidth and robust bias-corrected inference. They take your
Section 2 choices of running variable and controls. For each, write what it implies about the original claim.

**1. Both cutoffs.** The law has a second threshold at 0.15 with harsher punishment. Two cutoffs in one running
variable are two designs on one dataset: do they agree?

In [ ]:
# GIVEN — Section 3: rdrobust at a cutoff, with your running variable and controls
def rd(cutoff, h=None, kernel="triangular", controls=None, data=None, outcome="recidivism", donut=0.0):
    data = dwi if data is None else data
    d = data[(data[RUNNING] - cutoff).abs() >= donut]
    covs = d[controls] if controls else None
    kw = {"h": h} if h else {}
    r = rdrobust(y=d[outcome], x=d[RUNNING], c=cutoff, p=1, kernel=kernel, covs=covs, **kw)
    return {"jump": r.coef.iloc[0, 0], "se": r.se.iloc[0, 0], "p_robust": r.pv.iloc[2, 0],
            "h": r.bws.iloc[0, 0], "n": int(r.N_h[0] + r.N_h[1])}


def show(label, r):
    print(f"{label:<34} {r['jump']:+.4f} (SE {r['se']:.4f}, robust p {r['p_robust']:.3f}), h = {r['h']:.3f}, n = {r['n']:,}")


if RUNNING is None:
    print("Fill Section 2 first: RUNNING is not set yet.")
else:
    for c in (0.08, 0.15):
        show(f"cutoff {c}, MSE-optimal h", rd(c, controls=CONTROLS))

**2. Bandwidth.** The same jump at half, one and two times your Section 2 bandwidth, with your kernel. A design
that holds only at one bandwidth is a bandwidth, not a finding.

In [ ]:
# GIVEN — Section 3: the jump at H/2, H and 2H
if RUNNING is None or H is None:
    print("Fill Section 2 first: RUNNING and H are not set yet.")
else:
    k = "uniform" if KERNEL == "rectangular" else "triangular"
    for mult in (0.5, 1, 2):
        show(f"cutoff 0.08, h = {mult:g} x {H}", rd(CUTOFF, h=mult * H, kernel=k, controls=CONTROLS))

**3. Manipulation and balance.** Can drivers choose which side of 0.08 they land on? The density test asks whether
stops pile up on one side; the balance check asks whether drivers' characteristics jump at the cutoff, which they
should not if the cutoff is as good as random.

In [ ]:
# GIVEN — Section 3: the density test at 0.08, and each control as a placebo outcome
if RUNNING is None:
    print("Fill Section 2 first: RUNNING is not set yet.")
else:
    dens = rddensity(X=dwi[RUNNING].to_numpy(), c=CUTOFF)
    print(f"Density test at {CUTOFF}: t = {dens.test['t_jk']:+.3f}, p = {dens.test['p_jk']:.3f}")
    for v in (CONTROLS or []):
        show(f"balance: {v}", rd(CUTOFF, outcome=v))

**4. Your own specification.** Change at least one choice and justify it: the other cutoff, a donut (drop stops
within a small distance of the cutoff, where readings could be rounded), the other running variable, a different
kernel or bandwidth, or a subsample (first-time offenders by age, by year). `MY_SPEC` is a dictionary:
`{"cutoff": 0.08 or 0.15, "h": a bandwidth or None for MSE-optimal, "kernel": "uniform" or "triangular",
"donut": 0 or a distance, "sample": None or a pandas query string}`.

In [ ]:
# YOUR TURN — Section 3: your specification, and two sentences on why it is defensible
MY_SPEC = None
MY_SPEC_WHY = None

In [ ]:
# CHECK — Section 3: your own specification. Run it; you need not read it.
def check_my_spec(my_spec, baseline, why):
    if my_spec is None or why is None:
        print("Not filled yet: set MY_SPEC and MY_SPEC_WHY above.")
        return False
    if my_spec == baseline:
        print("MY_SPEC is the baseline. Change at least one choice, and say why that choice is defensible.")
        return False
    sentences = [s for s in str(why).replace("?", ".").split(".") if len(s.split()) >= 5]
    if len(sentences) < 2:
        print("MY_SPEC_WHY needs at least two sentences: what you changed, and why a careful author might have made that choice.")
        return False
    print("Specification accepted. Report it beside the baseline and say what the difference implies about the original claim.")
    return True

BASELINE = None if H is None else {"cutoff": 0.08, "h": H, "kernel": "uniform" if KERNEL == "rectangular" else "triangular",
                                   "donut": 0, "sample": None}
if check_my_spec(MY_SPEC, BASELINE, MY_SPEC_WHY):
    d = dwi.query(MY_SPEC["sample"]) if MY_SPEC["sample"] else dwi
    show("your specification", rd(MY_SPEC["cutoff"], h=MY_SPEC["h"], kernel=MY_SPEC["kernel"], controls=CONTROLS,
                                  data=d, donut=MY_SPEC["donut"]))
    print(f"Section 2 baseline: {JUMP:+.4f}")

**5. Your other exercises.** Section 3 asks for at least four. Natural ones here: a local quadratic beside the
local linear; placebo cutoffs where the law does nothing (0.05, 0.11); the jump by year, or by first versus repeat
stop; and whether the 0.08 and 0.15 jumps are the same size, and what it would mean if they were not. Write them
below this cell.

## Sections 4 to 6, in your repository

- **Section 4, does the finding hold?** Use your Section 3 results. "Holds in a narrower population", "holds at a
  smaller magnitude" and "holds under an assumption the authors did not state" are all answers.
- **Section 5, peer audit.** Assigned on the date the brief gives, never on your own paper; file the issue by the
  project deadline in the brief's Checkpoints list.
- **Section 6, your own repository.** Move the code you keep from this notebook into the template's layout, pin
  versions in `requirements.txt` (this notebook was built with pandas 2.2.2, numpy 2.0.2, statsmodels 0.14.6,
  linearmodels 7.1, rdrobust 2.0.0 and rddensity 3.0), and make the README name the one command that reproduces
  your headline table. Then clone your own repository into an empty folder and run that command.